# 第9章 浮动利率债券 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch09_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch09_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：例9.1 + 价格关于 DM 的曲线


In [ ]:
import numpy as np
from fi import frn, plotting
plotting.use_chinese_style()
for L in (0.01,0.02,0.03,0.04): print(f'基准={L:.0%}: 价格={frn.price_frn(L,0.005,0.005,8,4):.6f}')
dms = np.linspace(0, 0.02, 81)
fig, ax = plotting.new_axes()
ax.plot(dms*100, [frn.price_frn(0.02,0.005,dm,8,4) for dm in dms])
ax.axhline(100, ls=':', color='gray'); ax.axvline(0.5, ls=':', color='gray')
ax.set_xlabel('折现利差 DM (%)'); ax.set_ylabel('价格'); ax.set_title('DM=QM(0.5%) 处=面值'); fig.tight_layout()


## 编程实验 7：浮息 vs 固息的价格稳定性（图9-1）


In [ ]:
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
refs = np.linspace(0.01, 0.04, 61)
frn_p = [frn.price_frn(L,0.005,0.005,8,4) for L in refs]
fix_p = [price_bond(*make_cashflows(0.025,2,4,100), L+0.005, 4) for L in refs]
fig, ax = plotting.new_axes()
ax.plot(refs*100, frn_p, label='浮息债(DM=QM)'); ax.plot(refs*100, fix_p, label='固息债(票息2.5%)')
ax.axhline(100, ls=':', color='gray'); ax.set_xlabel('市场利率 (%)'); ax.set_ylabel('价格'); ax.set_title('浮息钉在面值,固息大幅波动'); ax.legend(); fig.tight_layout()


## 编程实验 8：反求 DM + 利差久期 vs 固息修正久期


In [ ]:
from fi import risk
dm = frn.discount_margin(99.50, 0.02, 0.005, 8, 4); print(f'市价99.50 反求 DM = {dm*100:.4f}%')
def spread_dur(ref,qm,d,n,f,bp=1e-4):
    p0=frn.price_frn(ref,qm,d,n,f); pu=frn.price_frn(ref,qm,d+bp,n,f); pd=frn.price_frn(ref,qm,d-bp,n,f)
    return (pd-pu)/(2*p0*bp)
print(f'浮息债利差久期 ≈ {spread_dur(0.02,0.005,0.005,8,4):.4f} 年')
cf,t = make_cashflows(0.025,2,4,100)
print(f'同期限固息债修正久期 ≈ {risk.modified_duration(cf,t,0.025,4):.4f} 年  (利差久期≈到期期限)')
